# run77 — band-power channel tokens (priority-BANDTOK)

**Uses the `-new` scripts** (`run_all_subjects-new.py`, `trial_level_eval-new.py`, `tsfnet_model_notebook-new.py`; keep all three in the same folder). The original scripts are untouched.

**Idea.** The existing `--eeg-spatial-mode mhsa` feeds attention a token that went through two strided convs (2000→40 samples, then *up*-sampled to 64), which cannot compute band power. Here each of the 28 channels is a token made of its **log band power** (δ, θ, α, β, γ × ten 1-s sub-windows = 50 numbers); one small pre-LN transformer block with zero-init residual gates; ~32k parameters (~8% of the model); joined to the EEG classifier through a zero-init Linear, so at step 0 the model **equals the baseline** (verified on CPU).

Needs `{subject}_eeg_channel_windows.npy` (the same file `mhsa` used). No new preprocessing.

Baseline seed 42: window **0.6074**, trial-level **0.6325**. Stop rule: needs ≥ +1 pp AND better in ≥ 16/26 subjects, otherwise reject.

## A — log band power (`none`)

In [ ]:
%run run_all_subjects-new.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --eeg-spatial-mode bandtok --bandtok-norm none

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

## B — relative band power (`relative`: band / sum of bands, drops overall amplitude drift)

In [ ]:
%run run_all_subjects-new.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --eeg-spatial-mode bandtok --bandtok-norm relative

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

## trial-level evaluation — repeat the SAME `--eeg-spatial-mode` / `--bandtok-norm`

In [ ]:
%run trial_level_eval-new.py --seed 42 --variant-hash <hash> --expected-window-acc <grand_42> --eeg-spatial-mode bandtok --bandtok-norm none

In [ ]:
%run trial_level_eval-new.py --seed 42 --variant-hash <hash> --expected-window-acc <grand_42> --eeg-spatial-mode bandtok --bandtok-norm relative

## seeds 999 / 2024 — only for the winner

In [ ]:
%run run_all_subjects-new.py --seed 999 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --eeg-spatial-mode bandtok --bandtok-norm <winner>

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
%run run_all_subjects-new.py --seed 2024 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --eeg-spatial-mode bandtok --bandtok-norm <winner>